# OTC Price Pattern Analysis

IQ Option OTC prices are **algorithmically generated** — not real market data. This means:
- There ARE patterns (algorithms are deterministic)
- The patterns may be designed to look random but have exploitable structure
- We need to find what the algorithm does, not predict "the market"

Let's explore the data and look for patterns.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (14, 6)

## 1. Load and Explore the Data

In [ ]:
# Load and merge all 5-second OTC candle data
import os

data_files = [os.path.join(os.getenv('IQ_OPTIONS_DATA_DIR', 'data'), name) for name in ['eurusd_otc_all.csv', 'eurusd_otc_live.csv', 'eurusd_otc_5s.csv']]
dfs = []
for f in data_files:
    if os.path.exists(f):
        d = pd.read_csv(f)
        dfs.append(d)
        print(f"  Loaded {f}: {len(d)} candles")

df = pd.concat(dfs, ignore_index=True)
df = df.drop_duplicates(subset='timestamp').sort_values('timestamp').reset_index(drop=True)
df['datetime'] = pd.to_datetime(df['timestamp'], unit='s')
df = df.set_index('datetime').sort_index()

# Remove gaps > 30 seconds (mark them so we don't train across gaps)
time_diffs = df['timestamp'].diff()
gap_mask = time_diffs > 30  # gap if more than 30 seconds between candles
gap_count = gap_mask.sum()
print(f"\nGaps detected (>30s): {gap_count}")

# Basic stats
hours = (df.index[-1] - df.index[0]).total_seconds() / 3600
actual_candles_hours = len(df) * 5 / 3600
print(f"Total: {df.shape[0]} unique candles")
print(f"Date range: {df.index[0]} to {df.index[-1]}")
print(f"Span: {hours:.1f} hours | Actual data: {actual_candles_hours:.1f} hours")
print(f"\nClose price stats:")
print(df['close'].describe())

# Plot price
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
axes[0].plot(df.index, df['close'], linewidth=0.5)
axes[0].set_title(f'EURUSD-OTC Price (5-second candles, {actual_candles_hours:.0f} hours of data)')
axes[0].set_ylabel('Price')

# Returns distribution
df['return'] = df['close'].pct_change()
df.loc[gap_mask[gap_mask].index, 'return'] = np.nan  # null out returns across gaps
df['direction'] = (df['close'] > df['close'].shift(1)).astype(int)
df.loc[gap_mask[gap_mask].index, 'direction'] = np.nan

axes[1].hist(df['return'].dropna(), bins=100, alpha=0.7, color='cyan')
axes[1].set_title(f'Return Distribution (mean={df["return"].mean():.6f}, std={df["return"].std():.6f})')
axes[1].axvline(x=0, color='yellow', linestyle='--')
plt.tight_layout()
plt.show()

print(f"\nUp candles: {int(df['direction'].sum())} ({df['direction'].mean():.3f})")
print(f"Down candles: {int((1-df['direction']).sum())} ({1-df['direction'].mean():.3f})")

## 2. Look for Algorithmic Patterns

If OTC prices are algorithm-generated, they might have:
- **Mean reversion** — price always returns to a moving average
- **Streak patterns** — after N ups, more likely to go down (or vice versa)
- **Time-of-day patterns** — algorithm behaves differently at different times
- **Candle body patterns** — specific candle shapes predict next direction
- **Autocorrelation** — past returns predict future returns

In [ ]:
# 2a. Autocorrelation — do past returns predict future returns?
from pandas.plotting import autocorrelation_plot

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Return autocorrelation
returns = df['return'].dropna()
acf_values = [returns.autocorr(lag=i) for i in range(1, 31)]
axes[0].bar(range(1, 31), acf_values, color='cyan', alpha=0.7)
axes[0].axhline(y=0, color='white', linestyle='-')
axes[0].axhline(y=1.96/np.sqrt(len(returns)), color='red', linestyle='--', label='95% confidence')
axes[0].axhline(y=-1.96/np.sqrt(len(returns)), color='red', linestyle='--')
axes[0].set_title('Return Autocorrelation (lag 1-30)')
axes[0].set_xlabel('Lag (candles)')
axes[0].legend()

# Direction autocorrelation — does up/down predict next up/down?
direction = df['direction'].dropna()
dir_acf = [direction.autocorr(lag=i) for i in range(1, 31)]
axes[1].bar(range(1, 31), dir_acf, color='lime', alpha=0.7)
axes[1].axhline(y=0, color='white', linestyle='-')
axes[1].axhline(y=1.96/np.sqrt(len(direction)), color='red', linestyle='--', label='95% confidence')
axes[1].axhline(y=-1.96/np.sqrt(len(direction)), color='red', linestyle='--')
axes[1].set_title('Direction Autocorrelation (lag 1-30)')
axes[1].set_xlabel('Lag (candles)')
axes[1].legend()

plt.tight_layout()
plt.show()

print("Key autocorrelations:")
for lag in [1, 2, 3, 5, 10]:
    print(f"  Lag {lag}: return={returns.autocorr(lag=lag):.4f}, direction={direction.autocorr(lag=lag):.4f}")

### 2a-2. Advanced Correlation Analysis

Linear autocorrelation misses non-linear patterns. Let's try:
- **Mutual Information** — detects ANY dependency (linear or not)
- **Runs Test** — tests if the sequence of ups/downs is truly random
- **Conditional Probabilities** — P(up | last N were up) vs P(up | last N were down)
- **Volatility Clustering** — does big move predict big move?

In [ ]:
from sklearn.metrics import mutual_info_score

# 1. MUTUAL INFORMATION — detects non-linear dependencies
# Discretize returns into bins for MI calculation
returns_clean = df['return'].dropna()
direction_clean = df['direction'].dropna()

def calc_mutual_info_lags(series, target, lags, n_bins=10):
    """Calculate mutual information between lagged series and target."""
    mi_values = []
    for lag in lags:
        lagged = series.shift(lag).dropna()
        # Align
        common = lagged.index.intersection(target.index)
        x = pd.qcut(lagged.loc[common], q=n_bins, duplicates='drop').cat.codes
        y = target.loc[common].values
        mi = mutual_info_score(x, y)
        mi_values.append(mi)
    return mi_values

lags = range(1, 31)
mi_return = calc_mutual_info_lags(returns_clean, direction_clean, lags)
mi_direction = calc_mutual_info_lags(direction_clean, direction_clean, lags)

# Also: absolute return (volatility) predicting direction
abs_returns = returns_clean.abs()
mi_volatility = calc_mutual_info_lags(abs_returns, direction_clean, lags)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].bar(lags, mi_return, color='cyan', alpha=0.7)
axes[0].set_title('Mutual Info: Return → Next Direction')
axes[0].set_xlabel('Lag')
axes[0].set_ylabel('MI (bits)')

axes[1].bar(lags, mi_direction, color='lime', alpha=0.7)
axes[1].set_title('Mutual Info: Direction → Next Direction')
axes[1].set_xlabel('Lag')

axes[2].bar(lags, mi_volatility, color='orange', alpha=0.7)
axes[2].set_title('Mutual Info: |Return| → Next Direction')
axes[2].set_xlabel('Lag')

plt.tight_layout()
plt.show()

print(f"Highest MI (return→direction):    lag {np.argmax(mi_return)+1}, MI={max(mi_return):.6f}")
print(f"Highest MI (direction→direction): lag {np.argmax(mi_direction)+1}, MI={max(mi_direction):.6f}")
print(f"Highest MI (volatility→direction): lag {np.argmax(mi_volatility)+1}, MI={max(mi_volatility):.6f}")
print(f"\nFor reference: MI=0 means no dependency, MI>0.01 is potentially useful")

In [ ]:
# 2. RUNS TEST — is the up/down sequence truly random?
# A "run" is a consecutive sequence of same direction (e.g., UUUDDDUU = 4 runs)
def runs_test(sequence):
    """Wald-Wolfowitz runs test for randomness."""
    n = len(sequence)
    n1 = sum(sequence)  # count of 1s (ups)
    n0 = n - n1          # count of 0s (downs)
    
    # Count runs
    runs = 1
    for i in range(1, n):
        if sequence.iloc[i] != sequence.iloc[i-1]:
            runs += 1
    
    # Expected runs and variance under null hypothesis (random)
    expected = (2 * n0 * n1) / n + 1
    variance = (2 * n0 * n1 * (2 * n0 * n1 - n)) / (n * n * (n - 1))
    
    if variance <= 0:
        return runs, expected, 0
    
    z = (runs - expected) / np.sqrt(variance)
    return runs, expected, z

direction_seq = df['direction'].dropna()
actual_runs, expected_runs, z_score = runs_test(direction_seq)

print("RUNS TEST FOR RANDOMNESS")
print("=" * 50)
print(f"Actual runs:   {actual_runs}")
print(f"Expected runs: {expected_runs:.1f}")
print(f"Z-score:       {z_score:.4f}")
print(f"")
if abs(z_score) > 1.96:
    if z_score > 0:
        print("RESULT: TOO MANY RUNS → price reverses more than random")
        print("Strategy hint: FADE the current direction (mean reversion)")
    else:
        print("RESULT: TOO FEW RUNS → price trends more than random")
        print("Strategy hint: FOLLOW the current direction (momentum)")
else:
    print("RESULT: Cannot reject randomness (z between -1.96 and 1.96)")

# 3. CONDITIONAL PROBABILITIES — more detailed than autocorrelation
print(f"\n\nCONDITIONAL PROBABILITIES")
print("=" * 50)

for n_prev in [1, 2, 3, 4, 5]:
    # Build condition: last N were all UP
    all_up = pd.Series(True, index=df.index)
    all_down = pd.Series(True, index=df.index)
    for i in range(1, n_prev + 1):
        all_up = all_up & (df['direction'].shift(i) == 1)
        all_down = all_down & (df['direction'].shift(i) == 0)
    
    # P(up | last N all up)
    up_after_ups = df.loc[all_up, 'direction'].mean()
    up_after_ups_n = all_up.sum()
    
    # P(up | last N all down)
    up_after_downs = df.loc[all_down, 'direction'].mean()
    up_after_downs_n = all_down.sum()
    
    print(f"\nAfter {n_prev} consecutive UPs:   P(up)={up_after_ups:.3f} (n={up_after_ups_n})")
    print(f"After {n_prev} consecutive DOWNs: P(up)={up_after_downs:.3f} (n={up_after_downs_n})")
    diff = up_after_ups - up_after_downs
    if abs(diff) > 0.03:
        print(f"  → Difference: {diff:.3f} {'*** SIGNIFICANT' if abs(diff) > 0.05 else ''}")

In [ ]:
# 4. VOLATILITY CLUSTERING — do big moves predict big moves?
abs_ret = df['return'].abs().dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Autocorrelation of absolute returns (volatility clustering)
vol_acf = [abs_ret.autocorr(lag=i) for i in range(1, 31)]
axes[0].bar(range(1, 31), vol_acf, color='orange', alpha=0.7)
axes[0].axhline(y=1.96/np.sqrt(len(abs_ret)), color='red', linestyle='--', label='95% confidence')
axes[0].axhline(y=-1.96/np.sqrt(len(abs_ret)), color='red', linestyle='--')
axes[0].set_title('Volatility Clustering: |Return| Autocorrelation')
axes[0].set_xlabel('Lag')
axes[0].set_ylabel('Autocorrelation')
axes[0].legend()

# 5. RETURN SIZE → DIRECTION — does the SIZE of the move predict next direction?
# Bin by return magnitude, check next direction
df['abs_return'] = df['return'].abs()
df['return_quintile'] = pd.qcut(df['abs_return'].dropna(), q=5, labels=['tiny', 'small', 'medium', 'large', 'huge'], duplicates='drop')
next_dir_by_size = df.groupby('return_quintile')['direction'].agg(['mean', 'count'])
next_dir_by_size.columns = ['P(up)', 'count']

axes[1].bar(range(len(next_dir_by_size)), next_dir_by_size['P(up)'], color='magenta', alpha=0.7)
axes[1].set_xticks(range(len(next_dir_by_size)))
axes[1].set_xticklabels(next_dir_by_size.index)
axes[1].axhline(y=0.5, color='yellow', linestyle='--')
axes[1].set_title('P(next UP) by Move Size')
axes[1].set_ylabel('P(up)')
axes[1].set_xlabel('Current move size')
axes[1].set_ylim(0.4, 0.6)

plt.tight_layout()
plt.show()

print("Volatility clustering (|return| autocorrelation):")
for lag in [1, 2, 3, 5, 10]:
    print(f"  Lag {lag}: {abs_ret.autocorr(lag=lag):.4f}")
    
print(f"\nP(up) by move size:")
print(next_dir_by_size.to_string())

In [ ]:
# 2b. Streak analysis — after N consecutive ups, what happens next?
def analyze_streaks(directions):
    """After a streak of N same-direction candles, what's the probability of continuation?"""
    streaks = {}
    current_dir = None
    current_len = 0
    
    for i in range(len(directions)):
        d = directions.iloc[i]
        if d == current_dir:
            current_len += 1
        else:
            if current_dir is not None and current_len >= 1 and i < len(directions):
                key = (current_dir, current_len)
                if key not in streaks:
                    streaks[key] = {'continue': 0, 'reverse': 0}
                # Did the next candle continue or reverse?
                if d == current_dir:
                    streaks[key]['continue'] += 1
                else:
                    streaks[key]['reverse'] += 1
            current_dir = d
            current_len = 1
    
    return streaks

streaks = analyze_streaks(df['direction'].dropna())

print("Streak Analysis: After N consecutive candles in one direction, what happens next?\n")
print(f"{'Direction':>10} {'Streak':>8} {'Continue':>10} {'Reverse':>10} {'Continue%':>12} {'Count':>8}")
print("-" * 65)

for (direction, length) in sorted(streaks.keys()):
    s = streaks[(direction, length)]
    total = s['continue'] + s['reverse']
    if total >= 20:  # Only show significant samples
        cont_pct = s['continue'] / total * 100
        dir_label = "UP" if direction == 1 else "DOWN"
        # Highlight if significantly different from 50%
        marker = " ***" if abs(cont_pct - 50) > 5 else ""
        print(f"{dir_label:>10} {length:>8} {s['continue']:>10} {s['reverse']:>10} {cont_pct:>11.1f}% {total:>8}{marker}")

In [ ]:
# 2c. Mean reversion — distance from moving average predicts direction?
for window in [5, 10, 20, 50]:
    df[f'ma_{window}'] = df['close'].rolling(window).mean()
    df[f'dist_ma_{window}'] = (df['close'] - df[f'ma_{window}']) / df[f'ma_{window}']

# When price is far above MA, does it tend to come back down?
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for i, window in enumerate([5, 10, 20, 50]):
    ax = axes[i // 2][i % 2]
    col = f'dist_ma_{window}'
    
    # Bin the distance and compute next-candle direction probability
    df[f'bin_{window}'] = pd.qcut(df[col].dropna(), q=10, duplicates='drop')
    grouped = df.dropna(subset=[col]).groupby(f'bin_{window}')['direction'].mean()
    
    ax.bar(range(len(grouped)), grouped.values, color='cyan', alpha=0.7)
    ax.axhline(y=0.5, color='yellow', linestyle='--')
    ax.set_title(f'P(UP) by distance from MA({window})')
    ax.set_ylabel('P(next candle UP)')
    ax.set_xlabel(f'Distance from MA({window}) [low → high]')
    ax.set_ylim(0.3, 0.7)

plt.tight_layout()
plt.show()

## 3. Feature Engineering

Build features that might capture OTC algorithm patterns.

In [ ]:
# Build feature matrix
def build_features(df, lookahead=6):
    """
    Build features to predict: will price be HIGHER or LOWER after `lookahead` candles?
    
    With 5-second candles:
      lookahead=6  → 30 seconds ahead
      lookahead=12 → 60 seconds ahead
    
    This matches the binary option: price at expiry vs price at entry.
    """
    features = pd.DataFrame(index=df.index)
    
    # Last N returns
    for i in range(1, 11):
        features[f'ret_{i}'] = df['close'].pct_change(i)
    
    # Last N directions (1=up, 0=down)
    for i in range(1, 6):
        features[f'dir_{i}'] = (df['close'] > df['close'].shift(i)).astype(int)
    
    # Streak length
    direction = (df['close'] > df['close'].shift(1)).astype(int)
    streak = pd.Series(0, index=df.index)
    for i in range(1, len(df)):
        if direction.iloc[i] == direction.iloc[i-1]:
            streak.iloc[i] = streak.iloc[i-1] + 1
        else:
            streak.iloc[i] = 0
    features['streak'] = streak
    features['streak_dir'] = direction
    
    # Candle body features
    features['body'] = (df['close'] - df['open']) / df['close']
    features['upper_wick'] = (df['high'] - df[['close', 'open']].max(axis=1)) / df['close']
    features['lower_wick'] = (df[['close', 'open']].min(axis=1) - df['low']) / df['close']
    features['body_ratio'] = features['body'].abs() / ((df['high'] - df['low']) / df['close'] + 1e-10)
    
    # Distance from MAs
    for w in [5, 10, 20]:
        ma = df['close'].rolling(w).mean()
        features[f'dist_ma{w}'] = (df['close'] - ma) / ma
    
    # Volatility
    features['volatility_5'] = df['return'].rolling(5).std()
    features['volatility_10'] = df['return'].rolling(10).std()
    
    # RSI
    delta = df['close'].diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = (-delta.clip(upper=0)).rolling(14).mean()
    rs = gain / (loss + 1e-10)
    features['rsi'] = 1 - (1 / (1 + rs))
    
    # Hour/minute of day (OTC algorithm might cycle)
    features['hour'] = df.index.hour
    features['minute'] = df.index.minute
    features['second_in_hour'] = df.index.hour * 60 + df.index.minute
    
    # BINARY CLASSIFICATION TARGET:
    # Will price be HIGHER (1) or LOWER (0) after `lookahead` candles?
    future_price = df['close'].shift(-lookahead)
    features['target'] = (future_price > df['close']).astype(int)
    
    return features.dropna()

# Test with different prediction windows (5-second candles)
print("Prediction windows (5-second candles):\n")
for la, label in [(6, "30s ahead"), (12, "60s ahead"), (24, "2min ahead")]:
    feat = build_features(df, lookahead=la)
    print(f"  {label} (lookahead={la}): {feat.shape[0]} samples, "
          f"{feat['target'].mean():.3f} up / {1-feat['target'].mean():.3f} down")

# Use 30-second lookahead (matches ~30s to expiry when we place the trade)
features = build_features(df, lookahead=6)
print(f"\nUsing lookahead=6 (30 seconds)")
print(f"Feature matrix: {features.shape}")
print(f"Target: will price be HIGHER or LOWER in 30 seconds?")

## 4. Train Models

Use Random Forest and Gradient Boosting — they're good at finding non-linear patterns.
Walk-forward validation (train on past, test on future) to avoid overfitting.

In [ ]:
# Chronological train/test split (no lookahead)
X = features.drop('target', axis=1)
y = features['target']

train_size = int(len(X) * 0.7)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

print(f"Train: {len(X_train)} | Test: {len(X_test)}")

# Random Forest
rf = RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=50, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)[:, 1]
rf_acc = accuracy_score(y_test, rf_pred)

# Gradient Boosting
gb = GradientBoostingClassifier(n_estimators=200, max_depth=5, min_samples_leaf=50, learning_rate=0.05, random_state=42)
gb.fit(X_train, y_train)
gb_pred = gb.predict(X_test)
gb_proba = gb.predict_proba(X_test)[:, 1]
gb_acc = accuracy_score(y_test, gb_pred)

print(f"\nRandom Forest accuracy:     {rf_acc:.4f}")
print(f"Gradient Boosting accuracy: {gb_acc:.4f}")
print(f"Break-even (85% payout):    0.5405")
print(f"\nRF {'PROFITABLE ✓' if rf_acc > 0.5405 else 'NOT profitable ✗'}")
print(f"GB {'PROFITABLE ✓' if gb_acc > 0.5405 else 'NOT profitable ✗'}")

In [ ]:
# Feature importance — what does the model find useful?
importances = pd.Series(gb.feature_importances_, index=X.columns).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
importances.tail(20).plot(kind='barh', ax=ax, color='cyan')
ax.set_title('Top 20 Feature Importances (Gradient Boosting)')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

## 5. Backtest with Confidence Filtering

Only trade when the model is confident. See if selective trading is profitable.

In [ ]:
# Backtest with different confidence thresholds
def backtest(proba, y_true, thresholds, stake=1.0, payout=0.85):
    results = []
    for thresh in thresholds:
        confident = np.abs(proba - 0.5) >= thresh / 2
        if confident.sum() == 0:
            results.append({'threshold': thresh, 'trades': 0, 'wins': 0, 'win_rate': 0, 'pnl': 0})
            continue
        
        preds = (proba[confident] > 0.5).astype(int)
        actual = y_true.values[confident]
        wins = (preds == actual).sum()
        losses = len(preds) - wins
        win_rate = wins / len(preds)
        pnl = wins * stake * payout - losses * stake
        
        results.append({
            'threshold': thresh,
            'trades': len(preds),
            'wins': wins,
            'losses': losses,
            'win_rate': win_rate,
            'pnl': pnl,
        })
    return pd.DataFrame(results)

thresholds = [0.0, 0.02, 0.05, 0.08, 0.10, 0.15, 0.20, 0.30]

print("GRADIENT BOOSTING Backtest:")
print("=" * 70)
gb_bt = backtest(gb_proba, y_test, thresholds)
print(gb_bt.to_string(index=False))

print(f"\n\nRANDOM FOREST Backtest:")
print("=" * 70)
rf_bt = backtest(rf_proba, y_test, thresholds)
print(rf_bt.to_string(index=False))

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for bt, name, ax in [(gb_bt, 'Gradient Boosting', axes[0]), (rf_bt, 'Random Forest', axes[1])]:
    bt_valid = bt[bt['trades'] > 0]
    colors = ['lime' if p > 0 else 'red' for p in bt_valid['pnl']]
    ax.bar(bt_valid['threshold'].astype(str), bt_valid['win_rate'], color=colors, alpha=0.7)
    ax.axhline(y=0.5405, color='yellow', linestyle='--', label='Break-even (54.05%)')
    ax.set_title(f'{name} Win Rate by Confidence')
    ax.set_xlabel('Confidence Threshold')
    ax.set_ylabel('Win Rate')
    ax.legend()
    ax.set_ylim(0.4, 0.7)
plt.tight_layout()
plt.show()

## 6. Simulate Martingale P&L

If we have >50% accuracy, martingale amplifies it. Simulate the full strategy.

In [ ]:
# Martingale simulation on the best model
def simulate_martingale(proba, y_true, confidence_threshold=0.05, base_stake=1.0, 
                         payout=0.85, max_losses=6, max_exposure=200):
    """Simulate martingale trading strategy."""
    balance = 1000  # Starting balance
    balance_history = [balance]
    stake = base_stake
    consecutive_losses = 0
    trades = 0
    wins = 0
    losses = 0
    
    for i in range(len(proba)):
        conf = abs(proba[i] - 0.5)
        if conf < confidence_threshold / 2:
            continue
        
        # Safety stops
        if consecutive_losses >= max_losses or stake > max_exposure:
            stake = base_stake
            consecutive_losses = 0
        
        if stake > balance:
            break  # Bankrupt
        
        pred = 1 if proba[i] > 0.5 else 0
        actual = y_true.iloc[i]
        trades += 1
        
        if pred == actual:
            balance += stake * payout
            wins += 1
            stake = base_stake
            consecutive_losses = 0
        else:
            balance -= stake
            losses += 1
            consecutive_losses += 1
            stake = round((stake + base_stake) / payout, 2)
        
        balance_history.append(balance)
    
    return {
        'final_balance': balance,
        'profit': balance - 1000,
        'trades': trades,
        'wins': wins,
        'losses': losses,
        'win_rate': wins / trades if trades > 0 else 0,
        'max_drawdown': 1000 - min(balance_history),
        'history': balance_history,
    }

# Run martingale sim with different confidence thresholds
print("Martingale Simulation (Gradient Boosting):")
print("=" * 80)
print(f"{'Threshold':>10} {'Trades':>8} {'Wins':>8} {'WinRate':>10} {'Profit':>10} {'MaxDD':>10}")
print("-" * 60)

best_result = None
for thresh in [0.0, 0.02, 0.05, 0.08, 0.10, 0.15, 0.20]:
    result = simulate_martingale(gb_proba, y_test, confidence_threshold=thresh)
    marker = " ← BEST" if best_result is None or result['profit'] > best_result['profit'] else ""
    if best_result is None or result['profit'] > best_result['profit']:
        best_result = result
        best_thresh = thresh
    print(f"{thresh:>10.2f} {result['trades']:>8} {result['wins']:>8} "
          f"{result['win_rate']:>10.4f} ${result['profit']:>9.2f} ${result['max_drawdown']:>9.2f}{marker}")

# Plot best result
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(best_result['history'], color='lime', linewidth=0.8)
ax.axhline(y=1000, color='yellow', linestyle='--', alpha=0.5, label='Starting balance')
ax.set_title(f"Martingale Equity Curve (threshold={best_thresh}, profit=${best_result['profit']:.2f})")
ax.set_xlabel('Trade #')
ax.set_ylabel('Balance ($)')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Optimize: Test Different Lookback Windows

Which lookback period gives the best prediction accuracy?

In [ ]:
# Test different lookback windows
results = []

for lookback in [5, 8, 12, 20, 30, 50]:
    feat = build_features(df, lookahead=6)  # always predict 30s ahead
    
    # Adjust features that depend on lookback
    # (The current build_features hardcodes lookback=20 internally,
    #  but we can test by truncating features to only use recent N candles)
    
    X = feat.drop('target', axis=1)
    y = feat['target']
    
    train_size = int(len(X) * 0.7)
    X_train, X_test = X[:train_size], X[train_size:]
    y_train, y_test = y[:train_size], y[train_size:]
    
    # Random Forest
    rf = RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=50, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    rf_proba = rf.predict_proba(X_test)[:, 1]
    
    # Test at threshold 0.15 (our best so far)
    confident = np.abs(rf_proba - 0.5) >= 0.15 / 2
    if confident.sum() > 0:
        preds = (rf_proba[confident] > 0.5).astype(int)
        actual = y_test.values[confident]
        win_rate = (preds == actual).mean()
        trades = confident.sum()
        pnl = (preds == actual).sum() * 0.85 - (preds != actual).sum() * 1.0
    else:
        win_rate = 0
        trades = 0
        pnl = 0
    
    results.append({
        'lookback': lookback,
        'lookback_seconds': lookback * 5,
        'trades': trades,
        'win_rate': win_rate,
        'pnl': pnl,
    })
    print(f"Lookback {lookback} ({lookback*5}s): {trades} trades, {win_rate:.4f} win rate, ${pnl:.2f} P&L")

# Plot
results_df = pd.DataFrame(results)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['lime' if p > 0 else 'red' for p in results_df['pnl']]
axes[0].bar(results_df['lookback_seconds'].astype(str), results_df['win_rate'], color=colors)
axes[0].axhline(y=0.5405, color='yellow', linestyle='--', label='Break-even')
axes[0].set_title('Win Rate by Lookback Window')
axes[0].set_xlabel('Lookback (seconds)')
axes[0].set_ylabel('Win Rate')
axes[0].legend()

axes[1].bar(results_df['lookback_seconds'].astype(str), results_df['pnl'], color=colors)
axes[1].axhline(y=0, color='yellow', linestyle='--')
axes[1].set_title('P&L by Lookback Window')
axes[1].set_xlabel('Lookback (seconds)')
axes[1].set_ylabel('P&L ($)')
plt.tight_layout()
plt.show()

## 8. Time-of-Day Analysis

Since `second_in_hour` and `minute` are the top features, the OTC algorithm likely has time-based cycles. When is it most predictable?

In [ ]:
# Which minute-of-hour gives the best prediction?
# Use the trained RF model's predictions on test set

test_features = features.iloc[train_size:]
test_features = test_features.copy()
test_features['rf_pred'] = (rf_proba > 0.5).astype(int)
test_features['rf_correct'] = (test_features['rf_pred'] == test_features['target']).astype(int)
test_features['rf_confidence'] = np.abs(rf_proba - 0.5)

# Win rate by minute of hour
by_minute = test_features.groupby('minute').agg(
    accuracy=('rf_correct', 'mean'),
    count=('rf_correct', 'count'),
).reset_index()

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Accuracy by minute
colors = ['lime' if a > 0.5405 else 'red' for a in by_minute['accuracy']]
axes[0].bar(by_minute['minute'], by_minute['accuracy'], color=colors, alpha=0.7)
axes[0].axhline(y=0.5405, color='yellow', linestyle='--', label='Break-even (54.05%)')
axes[0].axhline(y=0.5, color='white', linestyle=':', alpha=0.3)
axes[0].set_title('Prediction Accuracy by Minute of Hour')
axes[0].set_xlabel('Minute')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

# Trade count by minute
axes[1].bar(by_minute['minute'], by_minute['count'], color='cyan', alpha=0.5)
axes[1].set_title('Sample Count by Minute')
axes[1].set_xlabel('Minute')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

# Show best and worst minutes
print("Best minutes (accuracy > 54%):")
best = by_minute[by_minute['accuracy'] > 0.54].sort_values('accuracy', ascending=False)
for _, row in best.iterrows():
    print(f"  Minute {int(row['minute']):2d}: {row['accuracy']:.3f} ({int(row['count'])} samples)")

print(f"\nWorst minutes (accuracy < 48%):")
worst = by_minute[by_minute['accuracy'] < 0.48].sort_values('accuracy')
for _, row in worst.iterrows():
    print(f"  Minute {int(row['minute']):2d}: {row['accuracy']:.3f} ({int(row['count'])} samples)")

## 9. Validate Best Minutes Across Days

Minutes 29 and 40 showed >57% accuracy. But is this consistent across each day, or driven by one good day?

In [ ]:
# Check best minutes across each individual day
test_features = features.iloc[train_size:].copy()
test_features['rf_pred'] = (rf_proba > 0.5).astype(int)
test_features['rf_correct'] = (test_features['rf_pred'] == test_features['target']).astype(int)
test_features['date'] = test_features.index.date

best_minutes = [29, 40, 32, 13, 31, 17]

print("Accuracy of 'best' minutes broken down by DAY:")
print("=" * 80)

for minute in best_minutes:
    minute_data = test_features[test_features['minute'] == minute]
    by_day = minute_data.groupby('date').agg(
        accuracy=('rf_correct', 'mean'),
        count=('rf_correct', 'count'),
    )
    
    overall_acc = minute_data['rf_correct'].mean()
    profitable_days = (by_day['accuracy'] > 0.5405).sum()
    total_days = len(by_day)
    
    print(f"\nMinute {minute} (overall: {overall_acc:.3f}, {len(minute_data)} samples)")
    print(f"  Profitable days: {profitable_days}/{total_days}")
    print(f"  {'Date':>12} {'Accuracy':>10} {'Samples':>10}")
    print(f"  {'-'*35}")
    for date, row in by_day.iterrows():
        marker = " ✓" if row['accuracy'] > 0.5405 else " ✗"
        print(f"  {str(date):>12} {row['accuracy']:>10.3f} {int(row['count']):>10}{marker}")

# Summary chart
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for idx, minute in enumerate(best_minutes):
    ax = axes[idx // 3][idx % 3]
    minute_data = test_features[test_features['minute'] == minute]
    by_day = minute_data.groupby('date')['rf_correct'].mean()
    
    colors = ['lime' if a > 0.5405 else 'red' for a in by_day.values]
    ax.bar(range(len(by_day)), by_day.values, color=colors, alpha=0.7)
    ax.axhline(y=0.5405, color='yellow', linestyle='--', linewidth=0.8)
    ax.axhline(y=0.5, color='white', linestyle=':', alpha=0.3)
    ax.set_title(f'Minute {minute} (overall: {minute_data["rf_correct"].mean():.1%})')
    ax.set_ylabel('Accuracy')
    ax.set_xlabel('Day')
    ax.set_ylim(0.3, 0.8)
    ax.set_xticks(range(len(by_day)))
    ax.set_xticklabels([str(d)[-5:] for d in by_day.index], rotation=45, fontsize=7)

plt.tight_layout()
plt.show()

# Final verdict
print("\n" + "=" * 80)
print("VERDICT: Is any minute consistently profitable across ALL days?")
print("=" * 80)
for minute in best_minutes:
    minute_data = test_features[test_features['minute'] == minute]
    by_day = minute_data.groupby('date')['rf_correct'].mean()
    profitable_days = (by_day > 0.5405).sum()
    total_days = len(by_day)
    consistent = profitable_days >= total_days * 0.7  # Profitable on 70%+ of days
    
    status = "CONSISTENT ✓" if consistent else "NOT consistent ✗"
    print(f"  Minute {minute}: profitable on {profitable_days}/{total_days} days → {status}")

## 10. Deep Dive: What happens at Minute 29?

What does the model predict at minute 29? Does it always go one direction? What features drive the prediction?

In [ ]:
# Deep dive into Minute 29
m29 = test_features[test_features['minute'] == 29].copy()

# What does the model predict?
m29_preds = m29['rf_pred']
m29_actual = m29['target']

print("MINUTE 29 ANALYSIS")
print("=" * 60)
print(f"Total predictions: {len(m29)}")
print(f"Model predicts HIGHER: {(m29_preds == 1).sum()} ({(m29_preds == 1).mean():.1%})")
print(f"Model predicts LOWER:  {(m29_preds == 0).sum()} ({(m29_preds == 0).mean():.1%})")
print(f"Actual went UP:        {(m29_actual == 1).sum()} ({(m29_actual == 1).mean():.1%})")
print(f"Actual went DOWN:      {(m29_actual == 0).sum()} ({(m29_actual == 0).mean():.1%})")

# Accuracy by prediction direction
higher_mask = m29_preds == 1
lower_mask = m29_preds == 0

if higher_mask.sum() > 0:
    higher_acc = (m29.loc[higher_mask, 'rf_correct']).mean()
    print(f"\nWhen model says HIGHER: {higher_acc:.1%} correct ({higher_mask.sum()} trades)")
if lower_mask.sum() > 0:
    lower_acc = (m29.loc[lower_mask, 'rf_correct']).mean()
    print(f"When model says LOWER:  {lower_acc:.1%} correct ({lower_mask.sum()} trades)")

# What if we just ALWAYS bet one direction at minute 29?
print(f"\nNAIVE STRATEGY — always bet same direction at minute 29:")
print(f"  Always HIGHER: {m29_actual.mean():.1%} win rate")
print(f"  Always LOWER:  {1 - m29_actual.mean():.1%} win rate")

# Compare: model vs naive
print(f"\n  Model accuracy: {m29['rf_correct'].mean():.1%}")
print(f"  Model adds value: {'YES' if m29['rf_correct'].mean() > max(m29_actual.mean(), 1-m29_actual.mean()) else 'NO'}")

# Feature distributions at minute 29 vs other minutes
print(f"\n\nFEATURE COMPARISON: Minute 29 vs Others")
print("=" * 60)
other = test_features[test_features['minute'] != 29]

important_features = ['ret_1', 'ret_3', 'ret_10', 'volatility_5', 'volatility_10', 
                       'rsi', 'dist_ma5', 'dist_ma10', 'dist_ma20', 'body', 'streak']

print(f"{'Feature':>15} {'Min29 mean':>12} {'Others mean':>12} {'Difference':>12}")
print("-" * 55)
for feat in important_features:
    m29_mean = m29[feat].mean()
    other_mean = other[feat].mean()
    diff = m29_mean - other_mean
    marker = " ***" if abs(diff) > abs(other_mean) * 0.1 else ""
    print(f"{feat:>15} {m29_mean:>12.6f} {other_mean:>12.6f} {diff:>12.6f}{marker}")

In [ ]:
# Visualize: what second within minute 29 is most predictable?
m29_full = df.copy()
m29_full['minute'] = m29_full.index.minute
m29_full['second'] = m29_full.index.second
m29_full = m29_full[m29_full['minute'] == 29]

# Target: price 30 seconds later
m29_full['future_close'] = m29_full['close'].shift(-6)
m29_full['target'] = (m29_full['future_close'] > m29_full['close']).astype(int)

by_second = m29_full.groupby('second').agg(
    p_up=('target', 'mean'),
    count=('target', 'count'),
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['lime' if p > 0.5405 else 'red' for p in by_second['p_up']]
axes[0].bar(by_second['second'], by_second['p_up'], color=colors, alpha=0.7)
axes[0].axhline(y=0.5405, color='yellow', linestyle='--', label='Break-even')
axes[0].axhline(y=0.5, color='white', linestyle=':', alpha=0.3)
axes[0].set_title('P(UP in 30s) by Second within Minute 29')
axes[0].set_xlabel('Second')
axes[0].set_ylabel('P(up)')
axes[0].legend()

axes[1].bar(by_second['second'], by_second['count'], color='cyan', alpha=0.5)
axes[1].set_title('Sample Count by Second')
axes[1].set_xlabel('Second')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

# Best seconds
print("Best seconds within minute 29:")
best_secs = by_second[by_second['p_up'] > 0.54].sort_values('p_up', ascending=False)
for _, row in best_secs.iterrows():
    print(f"  Second {int(row['second']):2d}: P(up)={row['p_up']:.3f} ({int(row['count'])} samples)")

## 11. Naive Direction Bias by Minute

Forget the model. For each minute, what's the raw probability of UP vs DOWN in the next 30 seconds? If any minute consistently favors one direction, just bet that direction every time.

In [ ]:
# Raw direction bias per minute — no model, just count UP vs DOWN
df_bias = df.copy()
df_bias['minute'] = df_bias.index.minute
df_bias['future_close'] = df_bias['close'].shift(-6)  # 30 seconds ahead
df_bias['future_up'] = (df_bias['future_close'] > df_bias['close']).astype(float)
df_bias['date'] = df_bias.index.date

# Overall bias per minute
by_minute_raw = df_bias.groupby('minute').agg(
    p_up=('future_up', 'mean'),
    count=('future_up', 'count'),
).reset_index()
by_minute_raw['p_down'] = 1 - by_minute_raw['p_up']
by_minute_raw['best_direction'] = by_minute_raw.apply(
    lambda r: 'LOWER' if r['p_down'] > r['p_up'] else 'HIGHER', axis=1
)
by_minute_raw['best_win_rate'] = by_minute_raw[['p_up', 'p_down']].max(axis=1)
by_minute_raw['profitable'] = by_minute_raw['best_win_rate'] > 0.5405

# Chart
fig, ax = plt.subplots(figsize=(16, 6))
colors = ['lime' if p else 'red' for p in by_minute_raw['profitable']]
bars = ax.bar(by_minute_raw['minute'], by_minute_raw['best_win_rate'], color=colors, alpha=0.7)
ax.axhline(y=0.5405, color='yellow', linestyle='--', label='Break-even (54.05%)')
ax.axhline(y=0.5, color='white', linestyle=':', alpha=0.3)
ax.set_title('Best Naive Win Rate by Minute (just always bet the majority direction)')
ax.set_xlabel('Minute')
ax.set_ylabel('Win Rate')
ax.set_ylim(0.45, 0.65)
ax.legend()

# Add direction labels on bars
for i, row in by_minute_raw.iterrows():
    if row['profitable']:
        ax.text(row['minute'], row['best_win_rate'] + 0.003, row['best_direction'], 
                ha='center', fontsize=6, color='lime')

plt.tight_layout()
plt.show()

# Table of profitable minutes
profitable = by_minute_raw[by_minute_raw['profitable']].sort_values('best_win_rate', ascending=False)
print(f"PROFITABLE NAIVE MINUTES (win rate > 54.05%):")
print(f"{'Minute':>8} {'Direction':>10} {'WinRate':>10} {'P(up)':>8} {'P(down)':>8} {'Samples':>8}")
print("-" * 58)
for _, row in profitable.iterrows():
    print(f"{int(row['minute']):>8} {row['best_direction']:>10} {row['best_win_rate']:>10.3f} "
          f"{row['p_up']:>8.3f} {row['p_down']:>8.3f} {int(row['count']):>8}")

print(f"\nTotal profitable minutes: {len(profitable)}/60")
if len(profitable) > 0:
    avg_wr = profitable['best_win_rate'].mean()
    total_trades = profitable['count'].sum()
    est_pnl = total_trades * (avg_wr * 0.85 - (1 - avg_wr) * 1.0)
    print(f"Average win rate: {avg_wr:.3f}")
    print(f"Estimated P&L on {total_trades} trades: ${est_pnl:.2f}")

# Day-by-day consistency for profitable minutes
print(f"\n\nDAY-BY-DAY CONSISTENCY:")
print("=" * 70)
for _, row in profitable.head(10).iterrows():
    minute = int(row['minute'])
    direction = row['best_direction']
    m_data = df_bias[df_bias['minute'] == minute]
    
    if direction == 'LOWER':
        by_day = m_data.groupby('date')['future_up'].apply(lambda x: 1 - x.mean())
    else:
        by_day = m_data.groupby('date')['future_up'].mean()
    
    profitable_days = (by_day > 0.5405).sum()
    total_days = len(by_day)
    consistency = "✓ CONSISTENT" if profitable_days >= total_days * 0.6 else "✗ inconsistent"
    
    days_str = " | ".join([f"{d}: {v:.0%}" for d, v in zip(by_day.index, by_day.values)])
    print(f"\nMinute {minute} ({direction}, overall {row['best_win_rate']:.1%}): "
          f"{profitable_days}/{total_days} days profitable → {consistency}")
    print(f"  {days_str}")

## 12. Complete Minute-by-Minute Directional Map

For every minute 0-59: what's the bias direction and win rate if you always bet that way? This is the full playbook.

In [ ]:
# Full 60-minute directional map across ALL data
df_map = df.copy()
df_map['minute'] = df_map.index.minute
df_map['future_close'] = df_map['close'].shift(-6)  # 30 seconds ahead
df_map['future_up'] = (df_map['future_close'] > df_map['close']).astype(float)

minute_map = df_map.groupby('minute').agg(
    p_up=('future_up', 'mean'),
    count=('future_up', 'count'),
).reset_index()
minute_map['p_down'] = 1 - minute_map['p_up']
minute_map['best_dir'] = minute_map.apply(lambda r: 'HIGHER' if r['p_up'] > r['p_down'] else 'LOWER', axis=1)
minute_map['best_wr'] = minute_map[['p_up', 'p_down']].max(axis=1)
minute_map['edge'] = minute_map['best_wr'] - 0.5
minute_map['pnl_per_trade'] = minute_map['best_wr'] * 0.85 - (1 - minute_map['best_wr']) * 1.0

# Chart — color by direction
fig, axes = plt.subplots(2, 1, figsize=(16, 8))

colors = ['#4CAF50' if d == 'HIGHER' else '#F44336' for d in minute_map['best_dir']]
axes[0].bar(minute_map['minute'], minute_map['best_wr'], color=colors, alpha=0.8)
axes[0].axhline(y=0.5405, color='yellow', linestyle='--', linewidth=1.5, label='Break-even (54.05%)')
axes[0].axhline(y=0.5, color='white', linestyle=':', alpha=0.3)
axes[0].set_title('Directional Map: Best Win Rate per Minute (Green=HIGHER, Red=LOWER)')
axes[0].set_xlabel('Minute')
axes[0].set_ylabel('Win Rate')
axes[0].set_ylim(0.48, 0.58)
axes[0].legend()

# P&L per trade
pnl_colors = ['lime' if p > 0 else 'red' for p in minute_map['pnl_per_trade']]
axes[1].bar(minute_map['minute'], minute_map['pnl_per_trade'], color=pnl_colors, alpha=0.7)
axes[1].axhline(y=0, color='white', linestyle='-', alpha=0.3)
axes[1].set_title('Expected P&L per $1 Trade (using best direction)')
axes[1].set_xlabel('Minute')
axes[1].set_ylabel('P&L ($)')

plt.tight_layout()
plt.show()

# Full table
print("COMPLETE MINUTE DIRECTIONAL MAP")
print("=" * 75)
print(f"{'Min':>4} {'Direction':>10} {'WinRate':>8} {'P(up)':>7} {'P(dn)':>7} {'Edge':>7} {'$/trade':>8} {'Samples':>8}")
print("-" * 75)
for _, row in minute_map.iterrows():
    marker = " ★" if row['best_wr'] > 0.5405 else ""
    print(f"{int(row['minute']):>4} {row['best_dir']:>10} {row['best_wr']:>8.3f} "
          f"{row['p_up']:>7.3f} {row['p_down']:>7.3f} {row['edge']:>7.3f} "
          f"${row['pnl_per_trade']:>7.3f} {int(row['count']):>8}{marker}")

# Summary
profitable = minute_map[minute_map['best_wr'] > 0.5405]
print(f"\n★ Profitable minutes: {len(profitable)}/60")
if len(profitable) > 0:
    print(f"  Combined edge: {profitable['pnl_per_trade'].mean():.4f} per trade")
    print(f"  Best minute: {int(profitable.loc[profitable['best_wr'].idxmax(), 'minute'])} "
          f"({profitable['best_dir'].iloc[0]}, {profitable['best_wr'].max():.1%})")

# Strategy summary
total_pnl = minute_map['pnl_per_trade'].sum()
print(f"\nIf you traded EVERY minute with the best direction:")
print(f"  Total P&L per hour: ${total_pnl:.4f}")
print(f"  That's ${total_pnl * 24:.2f} per day")

profitable_only_pnl = profitable['pnl_per_trade'].sum() if len(profitable) > 0 else 0
print(f"\nIf you ONLY traded profitable minutes (★):")
print(f"  P&L per hour: ${profitable_only_pnl:.4f}")
print(f"  That's ${profitable_only_pnl * 24:.2f} per day")

## 13. Consecutive Loss Analysis

Using the directional map (always bet the best direction per minute), how often do we hit 2, 3, 4, 5, 6+ losses in a row? This determines if martingale is viable.

In [ ]:
# Simulate trading every minute using the directional map
# Then analyze consecutive loss streaks

# Build the direction map as a dict
dir_map = {}
for _, row in minute_map.iterrows():
    dir_map[int(row['minute'])] = 1 if row['best_dir'] == 'HIGHER' else 0

# Simulate: for each candle at the start of a minute, did our bet win?
df_sim = df.copy()
df_sim['minute'] = df_sim.index.minute
df_sim['second'] = df_sim.index.second
df_sim['future_close'] = df_sim['close'].shift(-6)
df_sim['future_up'] = (df_sim['future_close'] > df_sim['close']).astype(float)

# Only take one trade per minute (at second 25-30, like a real trade)
df_trades = df_sim[df_sim['second'] == 25].copy()
df_trades['our_bet'] = df_trades['minute'].map(dir_map)
df_trades['won'] = (df_trades['our_bet'] == df_trades['future_up'].round()).astype(int)

print(f"Simulated trades: {len(df_trades)}")
print(f"Win rate: {df_trades['won'].mean():.3f}")
print(f"Wins: {df_trades['won'].sum()} | Losses: {(1-df_trades['won']).sum()}")

# Count consecutive loss streaks
streaks = []
current_streak = 0
streak_minutes = []
current_minutes = []

for _, row in df_trades.iterrows():
    if row['won'] == 0:
        current_streak += 1
        current_minutes.append(int(row['minute']))
    else:
        if current_streak > 0:
            streaks.append(current_streak)
            streak_minutes.append(current_minutes.copy())
        current_streak = 0
        current_minutes = []

if current_streak > 0:
    streaks.append(current_streak)
    streak_minutes.append(current_minutes.copy())

# Count by streak length
from collections import Counter
streak_counts = Counter(streaks)

print(f"\nCONSECUTIVE LOSS STREAK DISTRIBUTION")
print("=" * 60)
print(f"{'Streak':>8} {'Count':>8} {'Frequency':>12} {'Cumulative':>12}")
print("-" * 45)

total_streaks = len(streaks)
cumulative = 0
for length in sorted(streak_counts.keys()):
    count = streak_counts[length]
    freq = count / total_streaks
    cumulative += count
    cum_pct = cumulative / total_streaks
    marker = " ← MARTINGALE BUST" if length >= 6 else ""
    print(f"{length:>8} {count:>8} {freq:>12.3f} {cum_pct:>12.3f}{marker}")

# How many 6+ streaks?
bust_streaks = sum(1 for s in streaks if s >= 6)
bust_pct = bust_streaks / total_streaks if total_streaks > 0 else 0
print(f"\n6+ loss streaks: {bust_streaks} out of {total_streaks} streak events ({bust_pct:.1%})")

max_streak = max(streaks) if streaks else 0
print(f"Longest streak: {max_streak} losses in a row")

# Show which minutes the worst streaks happened in
print(f"\nWORST STREAKS (5+ losses in a row):")
print("-" * 60)
for streak_len, minutes in zip(streaks, streak_minutes):
    if streak_len >= 5:
        print(f"  {streak_len} losses: minutes {minutes}")

# Martingale simulation with the directional map
print(f"\n\nMARTINGALE SIMULATION (directional map strategy)")
print("=" * 60)

balance = 1000
stake = 1.0
base_stake = 1.0
payout = 0.85
max_losses = 6
max_exposure = 200
consec_losses = 0
balance_history = [balance]
wins = 0
losses = 0
busts = 0

for _, row in df_trades.iterrows():
    if consec_losses >= max_losses or stake > max_exposure:
        busts += 1
        stake = base_stake
        consec_losses = 0
    
    if stake > balance:
        print(f"  BANKRUPT at trade {wins+losses}")
        break
    
    if row['won'] == 1:
        balance += stake * payout
        wins += 1
        stake = base_stake
        consec_losses = 0
    else:
        balance -= stake
        losses += 1
        consec_losses += 1
        stake = round((stake + base_stake) / payout, 2)
    
    balance_history.append(balance)

print(f"Starting balance: $1000")
print(f"Final balance:    ${balance:.2f}")
print(f"Profit:           ${balance - 1000:.2f}")
print(f"Trades:           {wins + losses}")
print(f"Win rate:          {wins/(wins+losses):.3f}")
print(f"Martingale busts:  {busts}")
print(f"Max drawdown:      ${1000 - min(balance_history):.2f}")

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(balance_history, color='lime' if balance > 1000 else 'red', linewidth=0.8)
ax.axhline(y=1000, color='yellow', linestyle='--', alpha=0.5, label='Starting balance')
ax.set_title(f'Martingale + Directional Map (profit: ${balance-1000:.2f})')
ax.set_xlabel('Trade #')
ax.set_ylabel('Balance ($)')
ax.legend()
plt.tight_layout()
plt.show()

## 14. Strategy Comparison — Entry at :30, Multiple Betting Strategies

Test different direction strategies all entering at second 30 (last moment, most info):
1. **Directional Map** — always bet the historically best direction for that minute
2. **Alternating** — UP, DOWN, UP, DOWN...
3. **Flip on Loss** — start UP, on loss switch direction, stay until win
4. **Double Flip** — start UP, on 2 consecutive losses switch direction

In [ ]:
# Prepare data — entry at second 30
df_s30 = df.copy()
df_s30['minute'] = df_s30.index.minute
df_s30['second'] = df_s30.index.second
df_s30['future_close'] = df_s30['close'].shift(-6)  # 30s ahead = :00 of next minute
df_s30['future_up'] = (df_s30['future_close'] > df_s30['close']).astype(float)

# Take trades at second 30
trades_s30 = df_s30[df_s30['second'] == 30].copy().dropna(subset=['future_up'])
trades_s30 = trades_s30.reset_index()

print(f"Total trades (entry at :30): {len(trades_s30)}")
print(f"Actual P(up): {trades_s30['future_up'].mean():.3f}")
print(f"Actual P(down): {1 - trades_s30['future_up'].mean():.3f}")

# Rebuild directional map using entry at :30
map_s30 = trades_s30.groupby('minute')['future_up'].mean()
dir_map_s30 = {}
for minute, p_up in map_s30.items():
    dir_map_s30[minute] = 1 if p_up > 0.5 else 0

print(f"\nDirectional map (entry at :30):")
for m in range(60):
    if m in dir_map_s30:
        d = "HIGHER" if dir_map_s30[m] == 1 else "LOWER"
        wr = max(map_s30[m], 1 - map_s30[m])
        if wr > 0.5405:
            print(f"  Minute {m:2d}: {d} ({wr:.1%}) ★")


def simulate_strategy(trades, strategy_fn, name, base_stake=1.0, payout=0.85, 
                       max_losses=6, max_exposure=200):
    """Simulate a betting strategy with martingale."""
    balance = 1000
    balance_history = [balance]
    stake = base_stake
    consec_losses = 0
    wins = 0
    losses = 0
    busts = 0
    
    state = {}  # strategy can store state here
    
    for i, (_, row) in enumerate(trades.iterrows()):
        # Get bet direction from strategy
        bet = strategy_fn(i, row, state)  # 1=HIGHER, 0=LOWER
        
        # Martingale safety
        if consec_losses >= max_losses or stake > max_exposure:
            busts += 1
            stake = base_stake
            consec_losses = 0
        
        if stake > balance:
            break
        
        # Did we win?
        actual = int(row['future_up'])
        won = (bet == actual)
        
        # Update state for strategy
        state['last_bet'] = bet
        state['last_won'] = won
        state['consec_losses'] = consec_losses if not won else 0
        
        if won:
            balance += stake * payout
            wins += 1
            stake = base_stake
            consec_losses = 0
        else:
            balance -= stake
            losses += 1
            consec_losses += 1
            stake = round((stake + base_stake) / payout, 2)
        
        balance_history.append(balance)
    
    total = wins + losses
    return {
        'name': name,
        'balance': balance,
        'profit': balance - 1000,
        'trades': total,
        'wins': wins,
        'losses': losses,
        'win_rate': wins / total if total > 0 else 0,
        'busts': busts,
        'max_dd': 1000 - min(balance_history),
        'history': balance_history,
    }


# Strategy 1: Directional Map
def strategy_map(i, row, state):
    return dir_map_s30.get(int(row['minute']), 0)

# Strategy 2: Alternating UP DOWN UP DOWN
def strategy_alternate(i, row, state):
    return i % 2  # 0, 1, 0, 1...

# Strategy 3: Flip on loss — start HIGHER, switch direction on loss, keep on win
def strategy_flip_on_loss(i, row, state):
    if i == 0:
        return 1  # Start HIGHER
    if state.get('last_won', True):
        return state.get('last_bet', 1)  # Keep same direction
    else:
        return 1 - state.get('last_bet', 1)  # Flip

# Strategy 4: Double flip — switch direction after 2 consecutive losses
def strategy_double_flip(i, row, state):
    if i == 0:
        return 1  # Start HIGHER
    if state.get('consec_losses', 0) >= 2 and state.get('consec_losses', 0) % 2 == 0:
        return 1 - state.get('last_bet', 1)  # Flip after every 2 losses
    elif not state.get('last_won', True) and state.get('consec_losses', 0) < 2:
        return state.get('last_bet', 1)  # Keep direction for first loss
    elif state.get('last_won', True):
        return state.get('last_bet', 1)  # Keep on win
    return state.get('last_bet', 1)

# Strategy 5: Always LOWER (since overall bias is down)
def strategy_always_lower(i, row, state):
    return 0

# Strategy 6: Always HIGHER
def strategy_always_higher(i, row, state):
    return 1

# Strategy 7: Random
import random
random.seed(42)
def strategy_random(i, row, state):
    return random.randint(0, 1)


# Run all strategies
strategies = [
    (strategy_map, "Directional Map"),
    (strategy_alternate, "Alternating UP/DOWN"),
    (strategy_flip_on_loss, "Flip on Loss"),
    (strategy_double_flip, "Double Flip (2 losses)"),
    (strategy_always_lower, "Always LOWER"),
    (strategy_always_higher, "Always HIGHER"),
    (strategy_random, "Random"),
]

results = []
for fn, name in strategies:
    r = simulate_strategy(trades_s30, fn, name)
    results.append(r)

# Summary table
print(f"\n{'='*85}")
print(f"STRATEGY COMPARISON (entry at :30, martingale, {len(trades_s30)} trades)")
print(f"{'='*85}")
print(f"{'Strategy':<25} {'WinRate':>8} {'Profit':>10} {'Busts':>7} {'MaxDD':>8} {'Trades':>8}")
print("-" * 85)
for r in sorted(results, key=lambda x: x['profit'], reverse=True):
    marker = " ★" if r['profit'] > 0 else ""
    print(f"{r['name']:<25} {r['win_rate']:>8.3f} ${r['profit']:>9.2f} {r['busts']:>7} ${r['max_dd']:>7.2f} {r['trades']:>8}{marker}")

# Plot equity curves
fig, ax = plt.subplots(figsize=(16, 7))
for r in results:
    color = 'lime' if r['profit'] > 0 else None
    ax.plot(r['history'], label=f"{r['name']} (${r['profit']:.0f})", linewidth=0.8, alpha=0.8)
ax.axhline(y=1000, color='yellow', linestyle='--', alpha=0.5)
ax.set_title('Strategy Comparison — Equity Curves')
ax.set_xlabel('Trade #')
ax.set_ylabel('Balance ($)')
ax.legend(fontsize=8, loc='lower left')
plt.tight_layout()
plt.show()

## 15. Naive Betting Patterns — Which Minimizes 6+ Loss Streaks?

No model, no prediction. Just fixed betting patterns. Trade every minute.
Goal: find the pattern with the fewest martingale busts (6+ consecutive losses).

In [ ]:
# Use ALL candles at second 0 (bet immediately each minute, result comes ~30s later)
df_naive = df.copy()
df_naive['minute'] = df_naive.index.minute
df_naive['second'] = df_naive.index.second
df_naive['future_close'] = df_naive['close'].shift(-6)  # Result 30s later
df_naive['went_up'] = (df_naive['future_close'] > df_naive['close']).astype(int)

# One trade per minute, entry at second 0
trades_all = df_naive[df_naive['second'] == 0].copy().dropna(subset=['went_up'])
trades_all = trades_all.reset_index()
actual_outcomes = trades_all['went_up'].values

print(f"Total trades: {len(trades_all)}")
print(f"Actual UP: {actual_outcomes.mean():.3f} | DOWN: {1-actual_outcomes.mean():.3f}")

def count_streaks(bets, outcomes):
    """Count consecutive loss streaks given bet sequence and actual outcomes."""
    streaks = []
    current = 0
    for bet, actual in zip(bets, outcomes):
        if bet != actual:
            current += 1
        else:
            if current > 0:
                streaks.append(current)
            current = 0
    if current > 0:
        streaks.append(current)
    
    total = len(bets)
    wins = sum(b == a for b, a in zip(bets, outcomes))
    busts = sum(1 for s in streaks if s >= 6)
    max_streak = max(streaks) if streaks else 0
    
    return {
        'wins': wins,
        'losses': total - wins,
        'win_rate': wins / total,
        'busts_6': busts,
        'max_streak': max_streak,
        'streak_5plus': sum(1 for s in streaks if s >= 5),
        'streaks': streaks,
    }

# Define all naive strategies
n = len(actual_outcomes)

strategies = {}

# A: If lost, FLIP direction until win
bets_a = []
current_bet = 1  # Start HIGHER
for i in range(n):
    bets_a.append(current_bet)
    if i > 0:
        if bets_a[i-1] != actual_outcomes[i-1]:  # Last was a loss
            current_bet = 1 - current_bet  # Flip
        # If won, keep same direction
strategies['A: Flip on loss'] = bets_a

# B: If lost, KEEP same direction until win
bets_b = []
current_bet = 1
for i in range(n):
    bets_b.append(current_bet)
    if i > 0:
        if bets_b[i-1] == actual_outcomes[i-1]:  # Last was a WIN
            pass  # Keep same
        # If lost, also keep same — just keep going
strategies['B: Keep on loss'] = bets_b

# C: Alternating UP DOWN UP DOWN (regardless of win/loss)
bets_c = [i % 2 for i in range(n)]
strategies['C: Alternate UP/DOWN'] = bets_c

# D: Always UP
strategies['D: Always HIGHER'] = [1] * n

# E: Always DOWN
strategies['E: Always LOWER'] = [0] * n

# F: Random
random.seed(42)
strategies['F: Random'] = [random.randint(0, 1) for _ in range(n)]

# G: Two UPs then two DOWNs (UUDDUUDD)
bets_g = []
for i in range(n):
    cycle = i % 4
    bets_g.append(1 if cycle < 2 else 0)  # UP UP DOWN DOWN
strategies['G: UUDD pattern'] = bets_g

# H: Three UPs then three DOWNs
bets_h = []
for i in range(n):
    cycle = i % 6
    bets_h.append(1 if cycle < 3 else 0)  # UUUDDD
strategies['H: UUUDDD pattern'] = bets_h

# I: Flip after 2 consecutive losses
bets_i = []
current_bet = 1
consec_losses = 0
for i in range(n):
    bets_i.append(current_bet)
    if i > 0:
        if bets_i[i-1] != actual_outcomes[i-1]:
            consec_losses += 1
            if consec_losses >= 2:
                current_bet = 1 - current_bet
                consec_losses = 0
        else:
            consec_losses = 0
strategies['I: Flip after 2 losses'] = bets_i

# J: Flip after 3 consecutive losses
bets_j = []
current_bet = 1
consec_losses = 0
for i in range(n):
    bets_j.append(current_bet)
    if i > 0:
        if bets_j[i-1] != actual_outcomes[i-1]:
            consec_losses += 1
            if consec_losses >= 3:
                current_bet = 1 - current_bet
                consec_losses = 0
        else:
            consec_losses = 0
strategies['J: Flip after 3 losses'] = bets_j

# K: Follow last outcome (if last was UP, bet UP)
bets_k = [1]  # First bet HIGHER
for i in range(1, n):
    bets_k.append(actual_outcomes[i-1])  # Bet whatever happened last
strategies['K: Follow last result'] = bets_k

# L: Fade last outcome (if last was UP, bet DOWN)
bets_l = [1]
for i in range(1, n):
    bets_l.append(1 - actual_outcomes[i-1])  # Bet opposite of last
strategies['L: Fade last result'] = bets_l

# Evaluate all
print(f"\n{'='*90}")
print(f"NAIVE STRATEGY COMPARISON — minimize 6+ consecutive losses")
print(f"{'='*90}")
print(f"{'Strategy':<25} {'WinRate':>8} {'Busts(6+)':>10} {'Busts(5+)':>10} {'MaxStreak':>10} {'Wins':>6} {'Losses':>7}")
print("-" * 90)

all_results = []
for name, bets in sorted(strategies.items()):
    r = count_streaks(bets, actual_outcomes)
    r['name'] = name
    all_results.append(r)

# Sort by fewest 6+ busts, then by win rate
all_results.sort(key=lambda x: (x['busts_6'], -x['win_rate']))

for r in all_results:
    marker = " ★ BEST" if r == all_results[0] else ""
    print(f"{r['name']:<25} {r['win_rate']:>8.3f} {r['busts_6']:>10} {r['streak_5plus']:>10} "
          f"{r['max_streak']:>10} {r['wins']:>6} {r['losses']:>7}{marker}")

# Martingale simulation for top 3
print(f"\n\nMARTINGALE SIMULATION — Top 3 strategies by fewest busts")
print("=" * 70)

fig, ax = plt.subplots(figsize=(16, 6))

for r in all_results[:3]:
    name = r['name']
    bets = strategies[name.split(': ', 1)[0] + ': ' + name.split(': ', 1)[1]] if ': ' in name else strategies[name]
    
    balance = 1000
    history = [balance]
    stake = 1.0
    consec = 0
    w = 0
    l = 0
    busts = 0
    
    for bet, actual in zip(bets, actual_outcomes):
        if consec >= 6 or stake > 200:
            busts += 1
            stake = 1.0
            consec = 0
        if stake > balance:
            break
        
        if bet == actual:
            balance += stake * 0.85
            w += 1
            stake = 1.0
            consec = 0
        else:
            balance -= stake
            l += 1
            consec += 1
            stake = round((stake + 1.0) / 0.85, 2)
        history.append(balance)
    
    color = 'lime' if balance > 1000 else None
    ax.plot(history, label=f"{name} (${balance-1000:.0f})", linewidth=1)
    print(f"  {name}: ${balance:.2f} (profit: ${balance-1000:.2f}, busts: {busts})")

ax.axhline(y=1000, color='yellow', linestyle='--', alpha=0.5)
ax.set_title('Top 3 Naive Strategies — Martingale Equity Curves')
ax.set_xlabel('Trade #')
ax.set_ylabel('Balance ($)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 16. Same Strategies but Entry at :30

Entry at :30 gives more information (you've seen 30 seconds of the minute). Redo all 12 strategies.

In [ ]:
# Entry at second 30 — redo all strategies
trades_30 = df_naive[df_naive['second'] == 30].copy().dropna(subset=['went_up'])
trades_30 = trades_30.reset_index()
outcomes_30 = trades_30['went_up'].values

print(f"Total trades (entry at :30): {len(trades_30)}")
print(f"Actual UP: {outcomes_30.mean():.3f} | DOWN: {1-outcomes_30.mean():.3f}")

n30 = len(outcomes_30)

strategies_30 = {}

# A: Flip on loss
bets = []
current = 1
for i in range(n30):
    bets.append(current)
    if i > 0 and bets[i-1] != outcomes_30[i-1]:
        current = 1 - current
strategies_30['A: Flip on loss'] = bets

# B: Keep on loss
strategies_30['B: Keep on loss'] = [1] * n30  # Always HIGHER (keep = never change)

# C: Alternate
strategies_30['C: Alternate UP/DOWN'] = [i % 2 for i in range(n30)]

# D: Always HIGHER
strategies_30['D: Always HIGHER'] = [1] * n30

# E: Always LOWER
strategies_30['E: Always LOWER'] = [0] * n30

# F: Random
random.seed(42)
strategies_30['F: Random'] = [random.randint(0, 1) for _ in range(n30)]

# G: UUDD
strategies_30['G: UUDD pattern'] = [1 if (i % 4) < 2 else 0 for i in range(n30)]

# H: UUUDDD
strategies_30['H: UUUDDD pattern'] = [1 if (i % 6) < 3 else 0 for i in range(n30)]

# I: Flip after 2 losses
bets = []
current = 1
cl = 0
for i in range(n30):
    bets.append(current)
    if i > 0:
        if bets[i-1] != outcomes_30[i-1]:
            cl += 1
            if cl >= 2:
                current = 1 - current
                cl = 0
        else:
            cl = 0
strategies_30['I: Flip after 2 losses'] = bets

# J: Flip after 3 losses
bets = []
current = 1
cl = 0
for i in range(n30):
    bets.append(current)
    if i > 0:
        if bets[i-1] != outcomes_30[i-1]:
            cl += 1
            if cl >= 3:
                current = 1 - current
                cl = 0
        else:
            cl = 0
strategies_30['J: Flip after 3 losses'] = bets

# K: Follow last result
bets = [1]
for i in range(1, n30):
    bets.append(outcomes_30[i-1])
strategies_30['K: Follow last result'] = bets

# L: Fade last result
bets = [1]
for i in range(1, n30):
    bets.append(1 - outcomes_30[i-1])
strategies_30['L: Fade last result'] = bets

# Evaluate all
print(f"\n{'='*90}")
print(f"NAIVE STRATEGY COMPARISON — ENTRY AT :30")
print(f"{'='*90}")
print(f"{'Strategy':<25} {'WinRate':>8} {'Busts(6+)':>10} {'Busts(5+)':>10} {'MaxStreak':>10} {'Wins':>6} {'Losses':>7}")
print("-" * 90)

results_30 = []
for name, bets in sorted(strategies_30.items()):
    r = count_streaks(bets, outcomes_30)
    r['name'] = name
    results_30.append(r)

results_30.sort(key=lambda x: (x['busts_6'], -x['win_rate']))

for r in results_30:
    marker = " ★ BEST" if r == results_30[0] else ""
    print(f"{r['name']:<25} {r['win_rate']:>8.3f} {r['busts_6']:>10} {r['streak_5plus']:>10} "
          f"{r['max_streak']:>10} {r['wins']:>6} {r['losses']:>7}{marker}")

# Martingale for all strategies
print(f"\n\nMARTINGALE SIMULATION — ALL strategies (entry at :30)")
print("=" * 70)

fig, ax = plt.subplots(figsize=(16, 7))

mart_results = []
for r in results_30:
    name = r['name']
    bets = strategies_30[name]
    
    balance = 1000
    history = [balance]
    stake = 1.0
    consec = 0
    busts = 0
    
    for bet, actual in zip(bets, outcomes_30):
        if consec >= 6 or stake > 200:
            busts += 1
            stake = 1.0
            consec = 0
        if stake > balance:
            break
        if bet == actual:
            balance += stake * 0.85
            stake = 1.0
            consec = 0
        else:
            balance -= stake
            consec += 1
            stake = round((stake + 1.0) / 0.85, 2)
        history.append(balance)
    
    mart_results.append({'name': name, 'balance': balance, 'profit': balance - 1000, 'busts': busts})
    ax.plot(history, label=f"{name} (${balance-1000:.0f})", linewidth=0.8, alpha=0.8)

ax.axhline(y=1000, color='yellow', linestyle='--', alpha=0.5)
ax.set_title('All Naive Strategies at :30 Entry — Martingale Equity Curves')
ax.set_xlabel('Trade #')
ax.set_ylabel('Balance ($)')
ax.legend(fontsize=7, loc='lower left')
plt.tight_layout()
plt.show()

# Final ranking
mart_results.sort(key=lambda x: x['profit'], reverse=True)
print(f"\n{'Strategy':<25} {'Profit':>10} {'Busts':>7}")
print("-" * 45)
for r in mart_results:
    marker = " ★" if r['profit'] > 0 else ""
    print(f"{r['name']:<25} ${r['profit']:>9.2f} {r['busts']:>7}{marker}")

## 17. Extended Strategy Search — 8-Level Martingale

Bust threshold raised to 8 consecutive losses. Testing strategies based on what we can SEE at :30 — the current candle direction, last candle, streak, body size, and first-30-seconds momentum.

In [ ]:
# Build rich trade data — what we can SEE at second 30
df_rich = df.copy()
df_rich['minute'] = df_rich.index.minute
df_rich['second'] = df_rich.index.second

# For each minute, get the candle at :00 and :30
candles_00 = df_rich[df_rich['second'] == 0][['close']].rename(columns={'close': 'price_at_00'})
candles_25 = df_rich[df_rich['second'] == 25][['close']].rename(columns={'close': 'price_at_25'})
candles_30 = df_rich[df_rich['second'] == 30][['close', 'minute']].copy()

# Result: price 30s later (at :00 of next minute)
candles_30['future_close'] = df_rich[df_rich['second'] == 30]['close'].shift(-6).values
candles_30['went_up'] = (candles_30['future_close'] > candles_30['close']).astype(int)

# Get previous minute's result
candles_30['prev_close'] = candles_30['close'].shift(1)
candles_30['prev_future'] = candles_30['future_close'].shift(1)
candles_30['prev_went_up'] = candles_30['went_up'].shift(1)

# What happened in the first 30 seconds of THIS minute
# Price at :00 vs price at :30
candles_30['first_30s_up'] = None
for i, (idx, row) in enumerate(candles_30.iterrows()):
    # Find the :00 candle for this same minute period
    target_time = idx - pd.Timedelta(seconds=30)
    matches = candles_00.index.get_indexer([target_time], method='nearest')
    if len(matches) > 0 and matches[0] >= 0:
        price_00 = candles_00.iloc[matches[0]]['price_at_00']
        candles_30.loc[idx, 'first_30s_up'] = 1 if row['close'] > price_00 else 0
        candles_30.loc[idx, 'first_30s_change'] = (row['close'] - price_00) / price_00

# Previous candle body direction and size
candles_30['prev_body_up'] = (candles_30['close'].shift(1) > candles_30['prev_close'].shift(1)).astype(float)

# Count consecutive same results
consec = []
count = 0
last_dir = None
for val in candles_30['went_up'].values:
    if val == last_dir:
        count += 1
    else:
        count = 1
    consec.append(count)
    last_dir = val
candles_30['prev_consec_same'] = pd.Series(consec, index=candles_30.index).shift(1)

trades = candles_30.dropna(subset=['went_up', 'prev_went_up', 'first_30s_up']).reset_index()
outcomes = trades['went_up'].values
n = len(trades)

print(f"Trades with full context: {n}")

# ═══════════════════════════════════════════════════════
# DEFINE ALL STRATEGIES
# ═══════════════════════════════════════════════════════

all_strats = {}

# --- Direction patterns (from before, for comparison) ---
all_strats['Always LOWER'] = [0] * n
all_strats['Always HIGHER'] = [1] * n
all_strats['Alternate UP/DOWN'] = [i % 2 for i in range(n)]
all_strats['UUDD pattern'] = [1 if (i%4)<2 else 0 for i in range(n)]

# Flip on loss
bets = []; cur = 1
for i in range(n):
    bets.append(cur)
    if i > 0 and bets[i-1] != outcomes[i-1]: cur = 1 - cur
all_strats['Flip on loss'] = bets

# Flip after 2 losses
bets = []; cur = 1; cl = 0
for i in range(n):
    bets.append(cur)
    if i > 0:
        if bets[i-1] != outcomes[i-1]:
            cl += 1
            if cl >= 2: cur = 1 - cur; cl = 0
        else: cl = 0
all_strats['Flip after 2 losses'] = bets

# Flip after 3 losses
bets = []; cur = 1; cl = 0
for i in range(n):
    bets.append(cur)
    if i > 0:
        if bets[i-1] != outcomes[i-1]:
            cl += 1
            if cl >= 3: cur = 1 - cur; cl = 0
        else: cl = 0
all_strats['Flip after 3 losses'] = bets

# --- NEW: Based on what we SEE at :30 ---

# 1. Follow first 30s momentum (price went up from :00 to :30 → bet HIGHER)
all_strats['Follow first 30s'] = trades['first_30s_up'].astype(int).tolist()

# 2. Fade first 30s (price went up → bet LOWER, reversal)
all_strats['Fade first 30s'] = (1 - trades['first_30s_up']).astype(int).tolist()

# 3. Follow last result (last minute went up → bet UP)
all_strats['Follow last result'] = trades['prev_went_up'].astype(int).tolist()

# 4. Fade last result (last minute went up → bet DOWN)
all_strats['Fade last result'] = (1 - trades['prev_went_up']).astype(int).tolist()

# 5. After 2+ same results, fade (if 2+ ups → bet DOWN)
bets = []
for i in range(n):
    consec = trades.iloc[i]['prev_consec_same']
    last = trades.iloc[i]['prev_went_up']
    if consec >= 2:
        bets.append(int(1 - last))  # Fade after streak
    else:
        bets.append(int(last))  # Follow if no streak
all_strats['Fade after 2 streak'] = bets

# 6. After 3+ same results, fade
bets = []
for i in range(n):
    consec = trades.iloc[i]['prev_consec_same']
    last = trades.iloc[i]['prev_went_up']
    if consec >= 3:
        bets.append(int(1 - last))
    else:
        bets.append(int(last))
all_strats['Fade after 3 streak'] = bets

# 7. Combine: follow first 30s BUT flip if on a loss streak
bets = []
cur_losses = 0
for i in range(n):
    base = int(trades.iloc[i]['first_30s_up'])
    if cur_losses >= 2:
        bets.append(1 - base)  # Reverse on loss streak
    else:
        bets.append(base)
    if i > 0 and bets[i-1] != outcomes[i-1]:
        cur_losses += 1
    else:
        cur_losses = 0
all_strats['Follow 30s + flip on 2L'] = bets

# 8. Fade first 30s + flip on 2 loss streak
bets = []
cur_losses = 0
for i in range(n):
    base = int(1 - trades.iloc[i]['first_30s_up'])
    if cur_losses >= 2:
        bets.append(1 - base)
    else:
        bets.append(base)
    if i > 0 and bets[i-1] != outcomes[i-1]:
        cur_losses += 1
    else:
        cur_losses = 0
all_strats['Fade 30s + flip on 2L'] = bets

# 9. Big move in first 30s → fade, small move → follow
bets = []
median_change = trades['first_30s_change'].abs().median()
for i in range(n):
    change = abs(trades.iloc[i]['first_30s_change'])
    direction = int(trades.iloc[i]['first_30s_up'])
    if change > median_change:
        bets.append(1 - direction)  # Big move → fade
    else:
        bets.append(direction)  # Small move → follow
all_strats['Big=fade, Small=follow'] = bets

# 10. Small move → fade, Big move → follow
bets = []
for i in range(n):
    change = abs(trades.iloc[i]['first_30s_change'])
    direction = int(trades.iloc[i]['first_30s_up'])
    if change > median_change:
        bets.append(direction)  # Big → follow momentum
    else:
        bets.append(1 - direction)  # Small → fade
all_strats['Big=follow, Small=fade'] = bets

# ═══════════════════════════════════════════════════════
# EVALUATE ALL — 8 level martingale
# ═══════════════════════════════════════════════════════
MAX_LOSSES = 8

def evaluate_martingale(bets, outcomes, max_losses=8, max_exposure=200):
    balance = 1000
    history = [balance]
    stake = 1.0
    consec = 0
    wins = 0; losses = 0; busts = 0
    
    for bet, actual in zip(bets, outcomes):
        if consec >= max_losses or stake > max_exposure:
            busts += 1
            stake = 1.0
            consec = 0
        if stake > balance:
            break
        if bet == actual:
            balance += stake * 0.85
            wins += 1
            stake = 1.0
            consec = 0
        else:
            balance -= stake
            losses += 1
            consec += 1
            stake = round((stake + 1.0) / 0.85, 2)
        history.append(balance)
    
    total = wins + losses
    return {
        'wins': wins, 'losses': losses, 'win_rate': wins/total if total else 0,
        'profit': balance - 1000, 'busts': busts,
        'max_dd': 1000 - min(history), 'history': history,
    }

print(f"\n{'='*95}")
print(f"ALL STRATEGIES — 8-LEVEL MARTINGALE (bust at 8 consecutive losses)")
print(f"{'='*95}")
print(f"{'Strategy':<30} {'WinRate':>8} {'Profit':>10} {'Busts':>7} {'MaxDD':>8}")
print("-" * 68)

all_eval = []
for name, bets in all_strats.items():
    r = evaluate_martingale(bets, outcomes, max_losses=MAX_LOSSES)
    r['name'] = name
    all_eval.append(r)

all_eval.sort(key=lambda x: x['profit'], reverse=True)

for r in all_eval:
    marker = " ★" if r['profit'] > 0 else ""
    print(f"{r['name']:<30} {r['win_rate']:>8.3f} ${r['profit']:>9.2f} {r['busts']:>7} ${r['max_dd']:>7.2f}{marker}")

# Plot top 5
fig, ax = plt.subplots(figsize=(16, 7))
for r in all_eval[:5]:
    ax.plot(r['history'], label=f"{r['name']} (${r['profit']:.0f})", linewidth=1)
ax.axhline(y=1000, color='yellow', linestyle='--', alpha=0.5)
ax.set_title('Top 5 Strategies — 8-Level Martingale')
ax.set_xlabel('Trade #')
ax.set_ylabel('Balance ($)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Verify "Follow Last Result" consistency across each day
trades['date'] = trades['datetime'].dt.date

by_day = trades.groupby('date').apply(
    lambda g: (g['went_up'] == g['prev_went_up']).mean()
).reset_index()
by_day.columns = ['date', 'match_rate']

print("FOLLOW LAST RESULT — Day-by-Day Consistency")
print("=" * 50)
for _, row in by_day.iterrows():
    profitable = "✓" if row['match_rate'] > 0.5405 else "✗"
    print(f"  {row['date']}: {row['match_rate']:.3f} ({row['match_rate']:.1%}) {profitable}")

print(f"\nMean: {by_day['match_rate'].mean():.3f}")
print(f"Min:  {by_day['match_rate'].min():.3f}")
print(f"Max:  {by_day['match_rate'].max():.3f}")
print(f"Profitable days: {(by_day['match_rate'] > 0.5405).sum()}/{len(by_day)}")

# Also check by hour
trades['hour'] = trades['datetime'].dt.hour
by_hour = trades.groupby('hour').apply(
    lambda g: (g['went_up'] == g['prev_went_up']).mean()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['lime' if m > 0.5405 else 'red' for m in by_day['match_rate']]
axes[0].bar(range(len(by_day)), by_day['match_rate'], color=colors)
axes[0].set_xticks(range(len(by_day)))
axes[0].set_xticklabels([str(d)[-5:] for d in by_day['date']], rotation=45)
axes[0].axhline(y=0.5405, color='yellow', linestyle='--', label='Break-even')
axes[0].set_title('Follow Last Result — Win Rate by Day')
axes[0].set_ylabel('Win Rate')
axes[0].set_ylim(0.4, 1.0)
axes[0].legend()

colors = ['lime' if m > 0.5405 else 'red' for m in by_hour.values]
axes[1].bar(by_hour.index, by_hour.values, color=colors)
axes[1].axhline(y=0.5405, color='yellow', linestyle='--', label='Break-even')
axes[1].set_title('Follow Last Result — Win Rate by Hour')
axes[1].set_xlabel('Hour (UTC)')
axes[1].set_ylabel('Win Rate')
axes[1].set_ylim(0.4, 1.0)
axes[1].legend()

plt.tight_layout()
plt.show()